# Network Traffic Anomaly Detection using an Autoencoder

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install scikit-learn pandas matplotlib seaborn tensorflow

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
col_names = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent",
             "hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root",
             "num_file_creations","num_shells","num_access_files","num_outbound_cmds","is_host_login",
             "is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate",
             "srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
             "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
             "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
             "dst_host_srv_rerror_rate","label","difficulty"]

df = pd.read_csv(url, names=col_names)
df.drop("difficulty", axis=1, inplace=True)
df['label'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
df = pd.get_dummies(df, columns=["protocol_type", "service", "flag"])
X = df.drop("label", axis=1)
y = df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train_ae = X_train[y_train == 0]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(X_train.shape[1], activation='linear')
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train_ae, X_train_ae, epochs=10, batch_size=256, validation_split=0.1, verbose=0)

reconstructions = model.predict(X_test)
mse = np.mean(np.square(X_test - reconstructions), axis=1)
threshold = np.percentile(mse, 95)
y_pred = (mse > threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
